# Lab 02 - Data Understanding: Impact of AI on Students

        Source dataset: `Datasets/Impact of AI on Students/ai_student_impact_dataset.csv`

        This notebook adapts the class lab pattern to the student-impact dataset. The source file is never modified.

        ## Lab concepts used

        - Summarise numeric and categorical fields.
- Visualise distributions and relationships.
- Identify modelling risks before preprocessing.

        Interpretation is predictive and associative only. The Kaggle source does not document how the records were collected or whether they represent observed students.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative} from {Path.cwd()}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns from {DATA_PATH}")

In [ ]:
IDENTIFIER = "Student_ID"
OUTCOMES = ["Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"]
EARLY_RISK_FEATURES = [
    "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case",
    "Prompt_Engineering_Skill", "Tool_Diversity", "Paid_Subscription",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Institutional_Policy",
]
EXPANDED_FEATURES = EARLY_RISK_FEATURES + ["Anxiety_Level_During_Exams"]

df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)

assert IDENTIFIER not in EARLY_RISK_FEATURES
assert not set(OUTCOMES).intersection(EARLY_RISK_FEATURES)
print("Leakage policy ready. Primary burnout model excludes anxiety and all post-semester outcomes.")

## Numeric and categorical summaries

In [ ]:
display(df.describe(include="number").T)
categorical_columns = [
    column for column in df.columns
    if pd.api.types.is_string_dtype(df[column])
    or pd.api.types.is_bool_dtype(df[column])
]
for column in categorical_columns:
    display(df[column].value_counts(dropna=False).rename("count").to_frame())

## Dataset-adapted Lab 2 visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.countplot(data=df, x="Burnout_Risk_Level", order=["Low", "Medium", "High"], ax=axes[0, 0])
sns.histplot(data=df, x="Skill_Retention_Score", bins=30, kde=True, ax=axes[0, 1])
sns.histplot(data=df, x="GPA_Change", bins=35, kde=True, ax=axes[1, 0])
sns.boxplot(data=df, x="Burnout_Risk_Level", y="Weekly_GenAI_Hours",
            order=["Low", "Medium", "High"], ax=axes[1, 1])
fig.tight_layout()
plt.show()

## Correlations and grouped comparisons

In [ ]:
numeric = df.select_dtypes(include="number").drop(columns=["Student_ID"])
plt.figure(figsize=(11, 8))
sns.heatmap(numeric.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Numeric-feature correlations")
plt.tight_layout()
plt.show()

display(
    df.groupby("Institutional_Policy", observed=True)[
        ["GPA_Change", "Skill_Retention_Score", "Weekly_GenAI_Hours"]
    ].agg(["mean", "median", "std"])
)

## What was learned from Lab 2

Pre-semester and post-semester GPA are strongly related, making raw post-GPA prediction too easy. Burnout is moderately imbalanced, while GPA decline is a clearer minority-class problem. Group differences are descriptive and cannot be interpreted as policy or AI effects.